# MLS211 Report Data Reader

## Task 1a

In [ ]:
# Import all necessary libraries
# Do this here so it is not necessary to run all sections sequentially
import pathlib

import pandas as pd

from mls211 import ExcelWriter, StandardCurveAnalyser, TableConverter, round_half_up

converter = TableConverter()

In [ ]:
# Read the copied task 1a data

DEFAULT_EXCEL_FILE = "./Task 1a/Task 1a.xlsx"

# S_Mean is the mean value submitted by the student
# S_Blanked is the blanked value submitted by the student
colnames = ["ID","Conc","Abs1", "Abs2", "S_Mean", "S_Blanked"]

# Copy submitted results to here
input_text = """
B 0.0 0.083 0.082 0.083 0.000
S1 0.1 0.091 0.122 0.107 0.024
S2 0.2 0.149 0.151 0.150 0.068
S3 0.5 1.127 1.119 1.123 1.041
S4 0.8 1.649 1.753 1.701 1.619
S5 1.0 1.717 2.083 1.900 1.818
"""

df = converter.text_to_pandas_dataframe(input_text.strip())
df.columns = colnames
analyser = StandardCurveAnalyser(df, 1,2,3)
df = analyser.calculate_std_curve()
reg = analyser.linear_regression(1,7)
r_squared = analyser.r_squared_forced_through_origin(1,7)
# Calculate concentration based on given absorbance
da_abs = 0.752
da_dilution = 5
da_conc = da_dilution*da_abs/reg[0]
# Round mean and blanked absorbances just prior to printing
# Round exactly to three decimal places
df['Mean_Abs'] = round_half_up(df['Mean_Abs'], "0.001")
df['Blanked_Abs'] = round_half_up(df['Blanked_Abs'], "0.001")
# Print output
print(df)
print(f"Slope: {reg[0]:.4f}, Intercept: {reg[1]:.4f}")
print(f"R-squared: {r_squared:.4f}")
print(f"Conc of unknown (abs={da_abs:.3f}): {da_conc:.1f} mmol/L")


In [ ]:
# Write to Excel
writer = ExcelWriter()
excel_output_path = DEFAULT_EXCEL_FILE
raw_data = df.iloc[:, [2,3]]
writer.write_dataframe_to_excel(raw_data, pathlib.Path(excel_output_path), "Table 3", 3, 6)
print(f"Data written to Excel file: {excel_output_path}")

## Task 1b

In [ ]:
# S_Mean is the mean value submitted by the student
# S_Blanked is the blanked value submitted by the student
colnames = ["ID","Conc","Abs1", "Abs2", "S_Mean", "S_Blanked"]

sample_dilution_factor = 16.01

# Absorbances for standard curve
std_data = """
RB 0.0 0.086 0.091 0.089 0.000
S1 0.1 0.286 0.302 0.294 0.206
S2 0.2 0.487 0.486 0.487 0.398
S3 0.5 1.062 1.083 1.073 0.984
S4 0.8 1.690 1.704 1.697 1.609
S5 1.0 2.091 2.146 2.119 2.030
"""

sample_data = """
P1-F Unknown 0.805 0.820 0.813 0.724
P1-60 Unknown 1.776 1.812 1.794 1.706
P1-120 Unknown 1.224 1.297 1.261 1.172
P2-F Unknown 1.019 1.036 1.028 0.939
P2-120 Unknown 1.539 1.527 1.533 1.445
QC Unknown 0.836 0.823 0.830 0.741
"""

df = converter.text_to_pandas_dataframe(std_data.strip())
df.columns = colnames
analyser = StandardCurveAnalyser(df, 1,2,3)
df = analyser.calculate_std_curve()
reg = analyser.linear_regression(1,7)
r_squared = analyser.r_squared_forced_through_origin(1,7)


df_samples = converter.text_to_pandas_dataframe(sample_data.strip())
df_samples.columns = colnames

df_samples = analyser.calculate_sample_concentration(df_samples,"Conc",2,3,"Blanked_Abs", sample_dilution_factor)

# Round data prior to printing
df['Mean_Abs'] = round_half_up(df['Mean_Abs'], "0.001")
df['Blanked_Abs'] = round_half_up(df['Blanked_Abs'], "0.001")

print("Standard Curve")
print(df)
print(f"Slope: {reg[0]:.4f}, Intercept: {reg[1]:.4f}")
print(f"R-squared: {r_squared:.4f}")

df_samples['Mean_Abs'] = round_half_up(df_samples['Mean_Abs'], "0.001")
df_samples['Blanked_Abs'] = round_half_up(df_samples['Blanked_Abs'], "0.001")
print("Samples")
print(df_samples)

In [ ]:
# Write to Task1b.xlsx
writer = ExcelWriter()
excel_output_path = "./Task 1b/Task1b.xlsx"
raw_data = df.iloc[:, [2,3]]
writer.write_dataframe_to_excel(raw_data, pathlib.Path(excel_output_path),"Sheet1", 3, 6)


raw_data_samples = df_samples.iloc[:, [2,3]]
writer.write_dataframe_to_excel(raw_data_samples, pathlib.Path(excel_output_path),"Sheet1", 3, 13)

df_slope = pd.DataFrame([reg[0]], columns=["Gradient"])
writer.write_dataframe_to_excel(df_slope, pathlib.Path(excel_output_path),"Sheet1", 6, 22)

print(f"Data written to Excel file: {excel_output_path}")